In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import mlflow
import mlflow.sklearn

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

c:\Users\chira\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_df = pd.read_csv("../data/processed/train.csv")
test_df = pd.read_csv("../data/processed/test.csv")

print("Train:", train_df.shape)
print("Test:", test_df.shape)

train_df.head()

Train: (5634, 22)
Test: (1409, 22)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,TenureGroup,ServiceCount,Churn
0,Male,0,No,No,35,No,No phone service,DSL,No,No,...,Yes,Yes,Month-to-month,No,Electronic check,49.20,1701.65,Established,3,0
1,Male,0,Yes,Yes,15,Yes,No,Fiber optic,Yes,No,...,No,No,Month-to-month,No,Mailed check,75.10,1151.55,Early,2,0
2,Male,0,Yes,Yes,13,No,No phone service,DSL,Yes,Yes,...,No,No,Two year,No,Mailed check,40.55,590.35,Early,3,0
3,Female,0,Yes,No,26,Yes,No,DSL,No,Yes,...,Yes,Yes,Two year,Yes,Credit card (automatic),73.50,1905.70,Established,5,0
4,Male,0,Yes,Yes,1,Yes,No,DSL,No,No,...,No,No,Month-to-month,No,Electronic check,44.55,44.55,New,1,0


In [3]:
X_train = train_df.drop(columns=["Churn", "customerID"], errors="ignore")
y_train = train_df["Churn"]

X_test = test_df.drop(columns=["Churn", "customerID"], errors="ignore")
y_test = test_df["Churn"]

In [5]:
numerical_features = [
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "ServiceCount"
]

categorical_features = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
    "TenureGroup"
]

In [6]:
required_features = numerical_features + categorical_features

missing = set(required_features) - set(X_train.columns)

if missing:
    raise ValueError(f"Missing features in training data: {missing}")

In [7]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features)
])

In [8]:
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        random_state=42,
        n_jobs=1
    ))
])

In [14]:
import mlflow

mlflow.set_tracking_uri("http://15.207.115.165:5000")

experiment = mlflow.set_experiment("telco-churn-tuning-v2")

print(experiment.name, experiment.experiment_id)

2026/10/09 17:47:30 INFO mlflow.tracking.fluent: Experiment with name 'telco-churn-tuning-v2' does not exist. Creating a new experiment.


telco-churn-tuning-v2 3


In [15]:
param_distributions = {
    "model__n_estimators": [200, 300, 500],
    "model__max_depth": [None, 8, 12, 16, 20],
    "model__min_samples_split": [2, 5, 10, 15],
    "model__min_samples_leaf": [1, 2, 4, 6],
    "model__max_features": ["sqrt", 0.7],
    "model__class_weight": [
        "balanced",
        "balanced_subsample"
    ]
}

In [16]:
from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_distributions,
    n_iter=25,
    scoring={
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1",
        "roc_auc": "roc_auc"
    },
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    refit="f1",
    return_train_score=False
)

In [ ]:
with mlflow.start_run(
    run_name="Random Forest Hyperparameter Tuning"
) as parent_run:

    mlflow.log_params({
        "model": "RandomForestClassifier",
        "search_method": "RandomizedSearchCV",
        "n_iter": 25,
        "cv_folds": 5,
        "selection_metric": "f1"
    })

    # Perform hyperparameter tuning once
    random_search.fit(X_train, y_train)

    cv_results = pd.DataFrame(random_search.cv_results_)

    # Log every trial as a separate nested run
    for i, row in cv_results.iterrows():

        with mlflow.start_run(
            run_name=f"RF Trial {i + 1:02d}",
            nested=True
        ):

            # Log hyperparameters
            mlflow.log_params(row["params"])

            # Log all five mean CV metrics
            for metric in [
                "accuracy",
                "precision",
                "recall",
                "f1",
                "roc_auc"
            ]:
                mlflow.log_metric(
                    f"cv_{metric}",
                    float(row[f"mean_test_{metric}"])
                )

                mlflow.log_metric(
                    f"cv_{metric}_std",
                    float(row[f"std_test_{metric}"])
                )

            mlflow.log_metric(
                "rank_f1",
                int(row["rank_test_f1"])
            )

    # Log best configuration on parent run
    mlflow.log_params({
        f"best_{key}": value
        for key, value in random_search.best_params_.items()
    })

    mlflow.log_metric(
        "best_cv_f1",
        float(random_search.best_score_)
    )

    # Save all search results
    cv_results.to_csv(
        "random_forest_cv_results.csv",
        index=False
    )

    mlflow.log_artifact("random_forest_cv_results.csv")

    print("Best CV F1:", random_search.best_score_)
    print("Best parameters:", random_search.best_params_)
    print("Parent run ID:", parent_run.info.run_id)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

best_model = random_search.best_estimator_

y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

test_metrics = {
    "test_accuracy": accuracy_score(y_test, y_pred),
    "test_precision": precision_score(
        y_test, y_pred, zero_division=0
    ),
    "test_recall": recall_score(
        y_test, y_pred, zero_division=0
    ),
    "test_f1": f1_score(
        y_test, y_pred, zero_division=0
    ),
    "test_roc_auc": roc_auc_score(y_test, y_prob)
}

with mlflow.start_run(run_name="Random Forest Tuned Evaluation"):

    mlflow.log_params(random_search.best_params_)
    mlflow.log_metric(
        "best_cv_f1",
        float(random_search.best_score_)
    )
    mlflow.log_metrics(test_metrics)

    mlflow.sklearn.log_model(
        best_model,
        name="tuned_random_forest"
    )

for metric, value in test_metrics.items():
    print(f"{metric}: {value:.4f}")

In [19]:
mlflow.set_tracking_uri("http://15.207.115.165:5000")

mlflow.set_experiment("telco-churn-gradient-boosting-tuning")

print("Experiment:", mlflow.get_experiment_by_name(
    "telco-churn-gradient-boosting-tuning"
).name)

2026/10/09 18:00:04 INFO mlflow.tracking.fluent: Experiment with name 'telco-churn-gradient-boosting-tuning' does not exist. Creating a new experiment.


Experiment: telco-churn-gradient-boosting-tuning


In [20]:
from sklearn.ensemble import GradientBoostingClassifier
gb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingClassifier(random_state=42))
])

In [21]:
param_distributions_gb = {
    "model__n_estimators": [100, 150, 200, 300],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1, 0.15],
    "model__max_depth": [2, 3, 4, 5],
    "model__min_samples_split": [2, 5, 10, 15],
    "model__min_samples_leaf": [1, 2, 4, 6],
    "model__subsample": [0.7, 0.8, 0.9, 1.0],
    "model__max_features": [None, "sqrt", "log2"]
}

In [22]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

random_search_gb = RandomizedSearchCV(
    estimator=gb_pipeline,
    param_distributions=param_distributions_gb,
    n_iter=25,
    scoring={
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1",
        "roc_auc": "roc_auc"
    },
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    refit="f1",
    return_train_score=False
)

In [ ]:
with mlflow.start_run(
    run_name="Gradient Boosting Hyperparameter Tuning"
) as parent_run:

    mlflow.log_params({
        "model": "GradientBoostingClassifier",
        "search_method": "RandomizedSearchCV",
        "n_iter": 25,
        "cv_folds": 5,
        "selection_metric": "f1"
    })

    random_search_gb.fit(X_train, y_train)

    cv_results_gb = pd.DataFrame(
        random_search_gb.cv_results_
    )

    for i, row in cv_results_gb.iterrows():

        with mlflow.start_run(
            run_name=f"GB Trial {i + 1:02d}",
            nested=True
        ):

            mlflow.log_params(row["params"])

            for metric in [
                "accuracy", "precision", "recall", "f1", "roc_auc"
            ]:
                mlflow.log_metric(
                    f"cv_{metric}",
                    float(row[f"mean_test_{metric}"])
                )

                mlflow.log_metric(
                    f"cv_{metric}_std",
                    float(row[f"std_test_{metric}"])
                )

            mlflow.log_metric(
                "rank_f1",
                int(row["rank_test_f1"])
            )

    mlflow.log_params({
        f"best_{key}": value
        for key, value in random_search_gb.best_params_.items()
    })

    mlflow.log_metric(
        "best_cv_f1",
        float(random_search_gb.best_score_)
    )

    cv_results_gb.to_csv(
        "gradient_boosting_cv_results.csv",
        index=False
    )

    mlflow.log_artifact(
        "gradient_boosting_cv_results.csv"
    )

print("Best CV F1:", random_search_gb.best_score_)
print("Best parameters:", random_search_gb.best_params_)

In [ ]:
best_gb_model = random_search_gb.best_estimator_

y_pred_gb = best_gb_model.predict(X_test)
y_prob_gb = best_gb_model.predict_proba(X_test)[:, 1]

test_metrics_gb = {
    "test_accuracy": accuracy_score(y_test, y_pred_gb),
    "test_precision": precision_score(
        y_test, y_pred_gb, zero_division=0
    ),
    "test_recall": recall_score(
        y_test, y_pred_gb, zero_division=0
    ),
    "test_f1": f1_score(
        y_test, y_pred_gb, zero_division=0
    ),
    "test_roc_auc": roc_auc_score(y_test, y_prob_gb)
}

with mlflow.start_run(
    run_name="Gradient Boosting Tuned Evaluation"
):

    mlflow.log_params(random_search_gb.best_params_)

    mlflow.log_metric(
        "best_cv_f1",
        float(random_search_gb.best_score_)
    )

    mlflow.log_metrics(test_metrics_gb)

    mlflow.sklearn.log_model(
        best_gb_model,
        name="tuned_gradient_boosting"
    )

print("\nGradient Boosting Test Results")

for metric, value in test_metrics_gb.items():
    print(f"{metric}: {value:.4f}")

In [37]:
import os
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# Numerical preprocessing
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features)
])

# Best Random Forest parameters
rf_model = RandomForestClassifier(
    n_estimators=200,
    min_samples_split=2,
    min_samples_leaf=4,
    max_features="sqrt",
    max_depth=8,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# Complete pipeline
best_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", rf_model)
])

# Train on training data
best_model.fit(X_train, y_train)

print("Final tuned Random Forest trained successfully!")

Final tuned Random Forest trained successfully!


In [38]:
os.makedirs("models", exist_ok=True)

joblib.dump(
    best_model,
    "models/tuned_random_forest.pkl"
)

print("Model saved successfully!")

Model saved successfully!
